# Feature exploration (Phase 1)

Interactive companion to `scripts/01_explore_pretrained_sae.py`. Runs locally or on **Colab/Kaggle** (free GPU).

On Colab, run the install cell first. Locally (with the repo's venv) you can skip it.

In [ ]:
# --- Colab/Kaggle setup (skip if running in the repo venv) ---
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformer-lens', 'sae-lens', 'datasets'], check=True)
    # If you cloned the repo into Colab, add its src/ to the path:
    # sys.path.insert(0, '/content/sae-interp/src')
print('IN_COLAB =', IN_COLAB)

In [ ]:
import torch
from sae_interp import net; net.bootstrap()
from sae_interp.config import load_config
from sae_interp.device import get_device
from sae_interp import features as F
from sae_interp.activations import load_corpus_tokens

cfg = load_config('../configs/sae_gpt2small.yaml')
device = get_device(); print('device', device)

In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE
model = HookedTransformer.from_pretrained(cfg.model.name, device=str(device)).eval()
sae = SAE.from_pretrained(cfg.pretrained_sae.release, cfg.pretrained_sae.sae_id, device=str(device))
if isinstance(sae, tuple): sae = sae[0]
print('d_sae', sae.cfg.d_sae)

In [ ]:
cfg.data.n_sequences = 300  # small for interactivity
tokens = load_corpus_tokens(cfg, model)
features_to_view = [0, 5, 12, 42, 100]
acts = F.collect_feature_activations(model, sae, tokens, cfg.model.hook_name, features_to_view, device)
for col, fi in enumerate(features_to_view):
    print(f'\n=== feature {fi} | density {F.feature_density(acts, col):.2%} ===')
    for ex in F.max_activating_examples(model, acts, tokens, col, top_k=8):
        print('  ', F.format_example(ex))